# 03 - Model Training and Evaluation

This notebook trains three models:

1. **Classification model**: predicts the probability of high surprise-billing risk  
2. **Regression model**: predicts estimated financial exposure  
3. **Multi-label source model**: predicts likely billing sources

Models are intentionally chosen to run on a normal laptop:
- Logistic Regression
- Random Forest
- HistGradientBoosting / Gradient Boosting

In [ ]:

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.multiclass import OneVsRestClassifier

pd.set_option("display.max_columns", 120)
os.makedirs("artifacts", exist_ok=True)

df = pd.read_parquet("data/model_features.parquet")
print(df.shape)
df.head()

In [ ]:

# Features and targets
y_class = df["proxy_surprise_bill_label"].astype(int)
y_reg = df["proxy_exposure_amount"].astype(float)

y_sources = df[["anesthesia_flag", "radiology_flag", "pathology_flag"]].astype(int)

numeric_features = [
    "average_covered_charges", "average_total_payments", "Billed amount", "Medicare payment",
    "charge_ratio_inpatient", "payment_gap_inpatient", "charge_ratio_outpatient", "payment_gap_outpatient",
    "blended_charge_ratio", "blended_payment_gap", "log_avg_covered", "log_avg_payments",
    "log_billed_amount", "log_medicare_payment", "log_blended_gap",
    "is_er", "is_imaging", "is_outpatient", "is_surgery", "high_complexity_drg",
    "provider_avg_gap", "provider_gap_std", "provider_avg_ratio", "provider_record_count",
    "state_avg_gap", "state_avg_ratio", "state_median_payment", "state_record_count",
    "provider_risk_index", "state_risk_index", "service_risk_weight",
]

categorical_features = ["provider_state", "service_type"]
text_feature = "drg_text"

X = df[numeric_features + categorical_features + [text_feature]].copy()

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

_, _, yreg_train, yreg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

_, _, ysrc_train, ysrc_test = train_test_split(
    X, y_sources, test_size=0.2, random_state=42
)

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("txt", TfidfVectorizer(max_features=100, ngram_range=(1, 2)), text_feature),
    ],
    remainder="drop"
)

In [ ]:

# Classification baselines
clf_log = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

clf_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=150,
        max_depth=10,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])

clf_gb = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])

for name, model in {"logistic": clf_log, "random_forest": clf_rf, "grad_boost": clf_gb}.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y_test, proba)
    print(f"\n{name} AUC: {auc:.4f}")
    print(classification_report(y_test, pred))

In [ ]:

# Pick the most presentation-friendly classifier
final_clf = clf_rf
final_clf.fit(X_train, y_train)

test_proba = final_clf.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

print("Final classification AUC:", roc_auc_score(y_test, test_proba))
print(confusion_matrix(y_test, test_pred))

In [ ]:

# Regression model
reg_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1
    ))
])

reg_model.fit(X_train, yreg_train)
reg_pred = reg_model.predict(X_test)

print("MAE:", mean_absolute_error(yreg_test, reg_pred))
print("RMSE:", mean_squared_error(yreg_test, reg_pred, squared=False))
print("R2:", r2_score(yreg_test, reg_pred))

In [ ]:

# Multi-label source prediction
source_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", OneVsRestClassifier(LogisticRegression(max_iter=1000)))
])

source_model.fit(X_train, ysrc_train)
source_pred = source_model.predict(X_test)

source_report = {}
for i, col in enumerate(y_sources.columns):
    source_report[col] = classification_report(ysrc_test.iloc[:, i], source_pred[:, i], output_dict=True)

pd.DataFrame({
    k: {
        "precision_1": v["1"]["precision"],
        "recall_1": v["1"]["recall"],
        "f1_1": v["1"]["f1-score"]
    }
    for k, v in source_report.items()
}).T

In [ ]:

# Feature importance from the Random Forest classifier
fitted_pre = final_clf.named_steps["preprocessor"]
fitted_model = final_clf.named_steps["model"]

feature_names = fitted_pre.get_feature_names_out()
importances = fitted_model.feature_importances_

fi = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)
fi.head(20)

In [ ]:

plt.figure(figsize=(10, 6))
top_fi = fi.head(15).sort_values("importance")
plt.barh(top_fi["feature"], top_fi["importance"])
plt.title("Top 15 Feature Importances - Classification Model")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:

# Save artifacts
joblib.dump(final_clf, "artifacts/risk_classifier.joblib")
joblib.dump(reg_model, "artifacts/exposure_regressor.joblib")
joblib.dump(source_model, "artifacts/source_predictor.joblib")
fi.to_csv("artifacts/feature_importance.csv", index=False)

metrics = {
    "classification_auc": float(roc_auc_score(y_test, test_proba)),
    "regression_mae": float(mean_absolute_error(yreg_test, reg_pred)),
    "regression_rmse": float(mean_squared_error(yreg_test, reg_pred, squared=False)),
    "regression_r2": float(r2_score(yreg_test, reg_pred)),
}

with open("artifacts/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Artifacts saved in artifacts/")